# Enhanced Federated Learning Cycle for DeepFake Detection

This notebook is **Flower-only** end-to-end.

Pipeline modules used:
- enhanced_client_selection.py
- update_validation.py
- knowledge_distillation.py
- client_reputation_ledger.py
- evaluation_metrics.py
- flwr_federated_cycle.py


In [2]:
# 1) Install dependencies and import Flower pipeline modules
import importlib
import importlib.util
import subprocess
import sys

def _pip_install(*packages):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"pip install failed: {' '.join(packages)}")

def _ensure_package(import_name: str, pip_spec: str) -> None:
    if importlib.util.find_spec(import_name) is None:
        _pip_install(pip_spec)

# TensorFlow + Flower simulation dependencies (Ray)
_ensure_package("tensorflow", "tensorflow-cpu>=2.16")
_ensure_package("ray", "ray[default]>=2.9")
_ensure_package("flwr", "flwr[simulation]>=1.7")

# Reload local module after code edits in the same notebook session
if "flwr_federated_cycle" in sys.modules:
    importlib.reload(sys.modules["flwr_federated_cycle"])

from flwr_federated_cycle import (
    FLWRCycleConfig,
    FLWRFederatedLearningCycle,
    generate_proxy_data,
    generate_synthetic_data,
    partition_data_iid_flwr,
)

print("Flower environment is ready.")


ModuleNotFoundError: No module named 'flwr_federated_cycle'

## Configuration

Set experiment hyperparameters for the Flower federated cycle.


In [ ]:
# Codespaces-friendly defaults to avoid OOM/kernel restarts with EfficientNet-B2
import os

IN_CODESPACES = os.environ.get("CODESPACES", "").lower() == "true"

# Keep non-Codespaces as the main target and derive a conservative Codespaces value.
NON_CODESPACE_GLOBAL_ROUNDS = 50
CODESPACES_MIN_ROUNDS = 5
CODESPACES_MAX_ROUNDS = 10
CODESPACES_GLOBAL_ROUNDS = max(
    CODESPACES_MIN_ROUNDS,
    min(CODESPACES_MAX_ROUNDS, NON_CODESPACE_GLOBAL_ROUNDS // 6),
)

if IN_CODESPACES:
    config = FLWRCycleConfig(
        model_path="efficientnetb4_final.keras",
        num_devices=20,
        local_epochs=1,
        global_rounds=CODESPACES_GLOBAL_ROUNDS,
        clients_per_round=4,
        local_batch_size=8,
        local_lr=1e-4,
        eval_every=5,
        enable_distillation=True,
        reports_dir="reports",
        checkpoints_dir="reports/checkpoints_flwr",
        auto_resume_from_checkpoint=True,
        simulation_client_cpus=1.0,
        simulation_client_gpus=0.0,
        simulation_local_mode=True,
        tflite_output_path="effnet_global_flwr_final.tflite",
    )
else:
    config = FLWRCycleConfig(
        model_path="efficientnetb4_final.keras",
        num_devices=100,
        local_epochs=5,
        global_rounds=NON_CODESPACE_GLOBAL_ROUNDS,
        clients_per_round=15,
        local_batch_size=32,
        local_lr=1e-4,
        eval_every=10,
        enable_distillation=True,
        reports_dir="reports",
        checkpoints_dir="reports/checkpoints_flwr",
        auto_resume_from_checkpoint=True,
        simulation_client_cpus=1.0,
        simulation_client_gpus=0.0,
        simulation_local_mode=False,
        tflite_output_path="effnet_global_flwr_final.tflite",
    )

print("Config created:")
print(f"  codespaces={IN_CODESPACES}")
print(f"  rounds={config.global_rounds}, clients={config.num_devices}, per_round={config.clients_per_round}, batch={config.local_batch_size}")
print(f"  auto_resume={config.auto_resume_from_checkpoint}, checkpoints_dir={config.checkpoints_dir}")
print(
    f"  rounds_policy: codespaces={CODESPACES_GLOBAL_ROUNDS} (from non-codespaces={NON_CODESPACE_GLOBAL_ROUNDS})"
)


Config created:
  codespaces=True
  rounds=8, clients=20, per_round=4, batch=8
  auto_resume=True, checkpoints_dir=reports/checkpoints_flwr
  rounds_policy: codespaces=8 (from non-codespaces=50)


## Data Preparation (Real FF++ TFRecords)

Use this section when your TFRecords are already prepared.

Where to place your links and paths:
1. In the next cell, edit USER INPUTS.
2. Set GOOGLE_DRIVE_TFRECORD_FOLDER_URL only for Codespaces or when you want auto-download.
3. Set COLAB_DRIVE_TFRECORD_ROOT to your folder inside MyDrive.
4. Set KAGGLE_TFRECORD_ROOT to your Kaggle dataset mount path.
5. If running Codespaces with synced files, set CODESPACE_LOCAL_TFRECORD_ROOT.

In [ ]:
# 2) Load real FF++ TFRecords from Google Drive / Kaggle / Codespaces
import os
import sys
import glob
import re
import subprocess
from pathlib import Path

import tensorflow as tf


# ========================= USER INPUTS =========================
# Put your shared Google Drive folder link here if you want auto-download.
# Example: https://drive.google.com/drive/folders/1AbCdEf...xyz
GOOGLE_DRIVE_TFRECORD_FOLDER_URL = ""

# Colab: folder path after mounting MyDrive
COLAB_DRIVE_TFRECORD_ROOT = "/content/drive/MyDrive/ffpp_tfrecord_clients"

# Kaggle: dataset mount path (after Add data)
KAGGLE_TFRECORD_ROOT = "/kaggle/input/ff-c23-tfrecord/ffpp_tfrecord_clients"

# Codespaces/local preferred path (optional override).
# Leave empty to auto-detect from workspace paths.
CODESPACE_LOCAL_TFRECORD_ROOT = ""

# Parsing and split config
def _resolve_model_input_shape() -> tuple[int, int]:
    try:
        from tensorflow.keras.applications.efficientnet import preprocess_input as _effnet_preprocess

        model = tf.keras.models.load_model(
            config.model_path,
            compile=False,
            custom_objects={"preprocess_input": _effnet_preprocess},
        )
        input_shape = model.input_shape
        if isinstance(input_shape, list):
            input_shape = input_shape[0]

        height = int(input_shape[1])
        width = int(input_shape[2])
        channels = int(input_shape[3]) if input_shape[3] is not None else 3

        config.input_shape = (height, width, channels)
        return (height, width)
    except Exception as exc:
        print(f"Warning: could not infer model input shape ({exc}). Using config.input_shape={config.input_shape}.")
        return tuple(config.input_shape[:2])

IMG_SIZE = _resolve_model_input_shape()
TFRECORD_GLOB = "client_*.tfrecord"
COMPRESSION_TYPE = "GZIP"
VAL_CLIENTS = 10
TEST_CLIENTS = 10
SHUFFLE_BUFFER = 2048
# Cap proxy size for distillation in resource-constrained environments.
# Set to 0 to disable cap.
PROXY_MAX_SAMPLES = 5000 if os.environ.get("CODESPACES", "").lower() == "true" else 0
EVAL_MAX_SAMPLES = 2000 if os.environ.get("CODESPACES", "").lower() == "true" else 0


def _in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


def _in_kaggle() -> bool:
    return os.path.isdir("/kaggle/input")


def _in_codespaces() -> bool:
    return (
        os.environ.get("CODESPACES", "").lower() == "true"
        or "GITHUB_CODESPACES_PORT_FORWARDING_DOMAIN" in os.environ
    )


def _candidate_workspace_roots() -> list[Path]:
    candidates: list[Path] = []

    # Explicit user override first.
    if CODESPACE_LOCAL_TFRECORD_ROOT.strip():
        candidates.append(Path(CODESPACE_LOCAL_TFRECORD_ROOT).expanduser())

    # Common workspace env vars.
    github_workspace = os.environ.get("GITHUB_WORKSPACE", "").strip()
    if github_workspace:
        candidates.append(Path(github_workspace) / "ffpp_tfrecord_clients")

    # Current working directory and nearby parents.
    cwd = Path.cwd()
    candidates.append(cwd / "ffpp_tfrecord_clients")
    candidates.append(cwd.parent / "ffpp_tfrecord_clients")

    # Typical Codespaces mount structure: /workspaces/<repo>/ffpp_tfrecord_clients
    for p in Path("/workspaces").glob("*/ffpp_tfrecord_clients"):
        candidates.append(p)

    # De-duplicate while preserving order.
    deduped: list[Path] = []
    seen: set[str] = set()
    for p in candidates:
        key = str(p)
        if key not in seen:
            seen.add(key)
            deduped.append(p)
    return deduped


def _find_existing_tfrecord_root() -> str | None:
    for p in _candidate_workspace_roots():
        if p.is_dir():
            matches = list(p.glob(TFRECORD_GLOB))
            if matches:
                return str(p)
    return None


def _download_tfrecord_folder(target_root: str) -> None:
    if not GOOGLE_DRIVE_TFRECORD_FOLDER_URL.strip():
        raise FileNotFoundError(
            "No TFRecord folder found in Codespaces workspace and "
            "GOOGLE_DRIVE_TFRECORD_FOLDER_URL is empty. "
            "Set CODESPACE_LOCAL_TFRECORD_ROOT or provide a Drive folder URL."
        )

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown>=5.0"])
    os.makedirs(target_root, exist_ok=True)

    subprocess.check_call([
        "gdown",
        "--folder",
        GOOGLE_DRIVE_TFRECORD_FOLDER_URL.strip(),
        "-O",
        target_root,
    ])


def resolve_tfrecord_root() -> str:
    if _in_colab():
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive", force_remount=False)
        root = COLAB_DRIVE_TFRECORD_ROOT
    elif _in_kaggle():
        root = KAGGLE_TFRECORD_ROOT
    else:
        detected_root = _find_existing_tfrecord_root()
        if detected_root is not None:
            root = detected_root
        else:
            # In Codespaces/local, if nothing is found, download into workspace.
            root = CODESPACE_LOCAL_TFRECORD_ROOT.strip() or "./ffpp_tfrecord_clients"
            _download_tfrecord_folder(root)

    if not os.path.isdir(root):
        raise FileNotFoundError(f"TFRecord root folder does not exist: {root}")
    return root


def parse_example(example_proto: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    feature_desc = {
        "image/encoded": tf.io.FixedLenFeature([], tf.string),
        "image/format": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.float32),
    }
    parsed = tf.io.parse_single_example(example_proto, feature_desc)

    image = tf.io.decode_jpeg(parsed["image/encoded"], channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)

    label = tf.cast(parsed["label"], tf.float32)
    return image, label


def load_client_dataset(tfrecord_path: str) -> tf.data.Dataset:
    ds = tf.data.TFRecordDataset(
        tfrecord_path,
        compression_type=COMPRESSION_TYPE,
        num_parallel_reads=tf.data.AUTOTUNE,
    )
    ds = ds.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
    return ds


def extract_client_id(path: str) -> str:
    name = Path(path).name
    match = re.search(r"client_(\d+)", name)
    if match:
        return str(int(match.group(1)))
    return name.replace(".tfrecord", "")


def concat_datasets(dataset_list: list[tf.data.Dataset]) -> tf.data.Dataset:
    if not dataset_list:
        raise ValueError("No datasets provided for concatenation.")
    combined = dataset_list[0]
    for ds in dataset_list[1:]:
        combined = combined.concatenate(ds)
    return combined


root = resolve_tfrecord_root()
all_files = sorted(glob.glob(os.path.join(root, TFRECORD_GLOB)))

if len(all_files) < 3:
    raise RuntimeError(
        f"Expected multiple TFRecord client shards at {root}, found {len(all_files)}"
    )

# Split by client shard: train / val / test
n_total = len(all_files)
n_val = min(VAL_CLIENTS, max(1, n_total // 10))
n_test = min(TEST_CLIENTS, max(1, n_total // 10))

val_files = all_files[:n_val]
test_files = all_files[n_val:n_val + n_test]
train_files = all_files[n_val + n_test:]

if not train_files:
    raise RuntimeError("No train client TFRecords left after val/test split. Reduce VAL_CLIENTS/TEST_CLIENTS.")

# Build federated client datasets
selected_train_files = train_files[: config.num_devices]
client_data = {
    str(i): load_client_dataset(fp).shuffle(SHUFFLE_BUFFER, seed=42)
    for i, fp in enumerate(selected_train_files)
}

config.num_devices = len(client_data)

# Build server validation and test datasets
server_val_data = concat_datasets([load_client_dataset(fp) for fp in val_files])
test_data = concat_datasets([load_client_dataset(fp) for fp in test_files])
if EVAL_MAX_SAMPLES > 0:
    server_val_data = server_val_data.take(EVAL_MAX_SAMPLES)
    test_data = test_data.take(EVAL_MAX_SAMPLES)

# Proxy data for KD: unlabeled stream from train clients
proxy_data = concat_datasets([client_data[cid].map(lambda image, label: image, num_parallel_calls=tf.data.AUTOTUNE) for cid in client_data])
if PROXY_MAX_SAMPLES > 0:
    proxy_data = proxy_data.take(PROXY_MAX_SAMPLES)

print(f"TFRecord root: {root}")
print(f"Image size used: {IMG_SIZE}")
print(f"Total client shards: {len(all_files)}")
print(f"Train/Val/Test shards: {len(selected_train_files)}/{len(val_files)}/{len(test_files)}")
print(f"Using num_devices={config.num_devices}")
if PROXY_MAX_SAMPLES > 0:
    print(f"Proxy cap enabled: {PROXY_MAX_SAMPLES} samples")
if EVAL_MAX_SAMPLES > 0:
    print(f"Eval cap enabled: {EVAL_MAX_SAMPLES} samples (val/test each)")


TFRecord root: /workspaces/Enhanced-Federated-Learning-Cycle-for-DeepFake-Detection/ffpp_tfrecord_clients
Image size used: (260, 260)
Total client shards: 100
Train/Val/Test shards: 20/10/10
Using num_devices=20
Proxy cap enabled: 5000 samples
Eval cap enabled: 2000 samples (val/test each)


## Build And Run Flower Cycle


In [ ]:
# 3) Build cycle and initialize model + modules
cycle = FLWRFederatedLearningCycle(config)
cycle.load_global_model()
cycle.create_clients(client_data)
cycle.setup_enhancement_modules()

print("Cycle initialized and ready to train.")


2026-03-17 01:10:20,690 | INFO     | Loading global model from efficientnetb4_final.keras ...
2026-03-17 01:10:23,274 | INFO     | Global model loaded -> 20,394,336 params, input shape (None, 260, 260, 3)
I0000 00:00:1773709823.300154   73475 tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144
2026-03-17 01:10:28,043 | INFO     | Created 20 federated clients.
2026-03-17 01:10:28,044 | INFO     | Enhancement modules (Parts 1–5) initialised.


Cycle initialized and ready to train.


In [ ]:
# 4) Run Flower federated learning
# Toggle this to temporarily force ultra-safe simulation settings in constrained environments.
USE_ULTRA_SAFE_SIM = False

run_server_val_data = server_val_data
run_test_data = test_data
run_proxy_data = proxy_data

if IN_CODESPACES and USE_ULTRA_SAFE_SIM:
    config.global_rounds = 1
    config.num_devices = 1
    config.clients_per_round = 1
    config.local_batch_size = 2
    config.simulation_client_cpus = 4.0  # 4 CPUs host -> 1 Ray actor
    config.simulation_client_gpus = 0.0
    config.simulation_local_mode = True

    # Keep only one client to avoid selector/sample mismatches.
    single_client = {k: client_data[k] for k in sorted(client_data.keys())[:1]}
    cycle.create_clients(single_client)
    cycle.config = config

    # Use tiny eval/proxy subsets for notebook debugging speed.
    run_server_val_data = server_val_data.take(64)
    run_test_data = test_data.take(128)
    run_proxy_data = proxy_data.take(128)

    print(
        "Using ultra-safe simulation settings:",
        f"rounds={config.global_rounds}, clients={config.num_devices}, ",
        f"per_round={config.clients_per_round}, batch={config.local_batch_size}, ",
        f"cpus/client={config.simulation_client_cpus}",
    )
else:
    print("Ultra-safe simulation settings are disabled.")

try:
    history = cycle.run(
        server_val_data=run_server_val_data,
        test_data=run_test_data,
        proxy_data=run_proxy_data,
    )
except RuntimeError as exc:
    print(f"Flower simulation failed: {exc}")
    print("Falling back to a local single-client training step for notebook continuity...")

    fallback_cid = sorted(cycle.client_datasets.keys())[0]
    fallback_train = (
        cycle.client_datasets[fallback_cid]
        .batch(config.local_batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )

    cycle.global_model.compile(
        optimizer=tf.keras.optimizers.Adam(config.local_lr),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    cycle.global_model.fit(fallback_train.take(1), epochs=1, verbose=0)

    eval_dict = cycle.global_model.evaluate(
        run_server_val_data.batch(config.local_batch_size),
        verbose=0,
        return_dict=True,
    )
    fallback_acc = float(eval_dict.get("accuracy", 0.0))

    history = {
        "round": [1],
        "enhanced_accuracy": [fallback_acc],
        "selected_clients": [[fallback_cid]],
        "num_accepted": [1],
        "num_rejected": [0],
        "distillation_loss": [None],
        "fallback_mode": [True],
    }
    print(f"Fallback completed. Approx server-val accuracy: {fallback_acc:.4f}")

print("Training complete.")
print(f"Rounds completed: {len(history.get('round', []))}")

2026-03-17 01:10:28,059 | INFO     | Auto-resume: no checkpoint found, starting fresh run.
2026-03-17 01:10:28,060 | INFO     | FL Cycle (Flower): 20 devices, 8 rounds to run (8 target, offset=0), 1 local epochs
2026-03-17 01:10:28,060 | INFO     | Starting evaluation for 'effnet_global_flwr' …


Ultra-safe simulation settings are disabled.


2026-03-17 01:15:15,189 | INFO     | Classification — Acc: 0.4785 | F1-macro: 0.4301 | ROC-AUC: 0.4932
2026-03-17 01:20:04,823 | INFO     | Latency — mean: 1158.20 ms | p95: 1301.22 ms | p99: 1533.95 ms
2026-03-17 01:20:04,903 | INFO     | Model size — params: 20,394,336 | disk: 77.80 MB
2026-03-17 01:20:04,906 | INFO     | Reports saved → effnet_global_flwr_20260317_012004_round_000_baseline_flwr.json  &  effnet_global_flwr_20260317_012004_round_000_baseline_flwr.txt
2026-03-17 01:20:04,906 | INFO     | Baseline -> Acc: 0.4785, F1: 0.4301, AUC: 0.4932
2026-03-17 01:20:07,254	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated

:job_id:01000000
:actor_name:ClientAppActor
:actor_name:ClientAppActor
:actor_name:ClientAppActor
:actor_name:ClientAppActor


INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
2026-03-17 01:20:17,395 | INFO     | Round 1 — selected 4 / 20 clients: ['0', '6', '4', '17']
INFO :      configure_fit: strategy sampled 4 clients (out of 20)
/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/backend/tensorflow/core.py:171: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.array(x)
ERROR :     Traceback (most recent call last):
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/flwr/simulation/ray_transport/ray_client_proxy.py", line 84, in _submit_job
    self.actor_pool.

Flower simulation failed: Flower simulation completed with zero successful rounds. In constrained environments, reduce clients_per_round/num_devices, lower simulation_client_cpus, or set simulation_local_mode=True. Last Flower client failure: ValueError("Invalid type of object refs, <class 'NoneType'>, is given. 'object_refs' must either be an ObjectRef or a list of ObjectRefs. ")
Falling back to a local single-client training step for notebook continuity...


/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/backend/tensorflow/core.py:171: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.array(x)
/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


Fallback completed. Approx server-val accuracy: 0.4845
Training complete.
Rounds completed: 1


In [ ]:
# 5) Quick summary
if history.get("enhanced_accuracy"):
    best_acc = max(history["enhanced_accuracy"])
    final_acc = history["enhanced_accuracy"][-1]
    print(f"Best enhanced accuracy:  {best_acc:.4f}")
    print(f"Final enhanced accuracy: {final_acc:.4f}")

print(f"TFLite output: {config.tflite_output_path}")
print("Reports dir:  reports/")


Best enhanced accuracy:  0.4845
Final enhanced accuracy: 0.4845
TFLite output: effnet_global_flwr_final.tflite
Reports dir:  reports/
